# Hybrid Retrieval Ablation — Người 3 (Kiệt) — v3

**Notebook chạy trên Kaggle** — Đánh giá Dense (chunk+metadata), BM25, Hybrid (Dense+BM25).

| # | Config | Dense | BM25 | Mục tiêu |
|---|--------|-------|------|----------|
| 1 | `Dense-ChunkMeta` | ✅ | ❌ | Dense baseline — chunk_text + metadata |
| 2 | `BM25-Only` | ❌ | ✅ | BM25 sparse baseline |
| 3 | `Retrieval-Hybrid-SparseDense` | ✅ | ✅ | Hybrid — score merge + dedup |

**BM25 sharded search**: Search từng shard riêng rồi merge kết quả (mỗi shard index độc lập).

---

## 1. Install & Imports

In [ ]:
!pip install -q rank_bm25 sentence-transformers faiss-cpu underthesea

In [ ]:
import json, os, sys, time, math, pickle, re, unicodedata, gc
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any
from collections import defaultdict
from datetime import datetime, timezone
import numpy as np
print('Setup complete.')

## 2. Configuration

In [ ]:
# ============================================================
# KAGGLE PATHS — CHỈNH LẠI NẾU TÊN KHÁC
# ============================================================
IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    FAISS_DIR = Path('/kaggle/input/datasets/kittrntunk/faiss-chunk-meta')
    BM25_BASE_DIR = Path('/kaggle/input/datasets/nguyenlethienlyy/bm25-tokenized/bm25')
    QA_DIR = Path('/kaggle/input/datasets/phuongthao205/qa-legalrag/Benchmark')
    OUTPUT_BASE = Path('/kaggle/working/evaluation_runs/ablation')
else:
    PROJECT_ROOT = Path('.').resolve()
    if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    FAISS_DIR = PROJECT_ROOT / 'data' / 'faiss_index'
    BM25_BASE_DIR = PROJECT_ROOT / 'data' / 'sparse_index'
    QA_DIR = PROJECT_ROOT / 'data' / 'benchmark'
    OUTPUT_BASE = PROJECT_ROOT / 'evaluation_runs' / 'ablation'

EMBEDDING_MODEL = 'intfloat/multilingual-e5-large'
TOP_K = 30
TOP_N = 10
SCORE_THRESHOLD = 0.30
TOP_K_EVAL = [1, 3, 5, 10]
ABLATION_LIMIT = None  # Set 10 để smoke test

os.environ['CUDA_VISIBLE_DEVICES'] = ''  # Force CPU — tránh lỗi GPU
# === GENERATOR ===
LLM_BASE_URL = 'https://api.shopaikey.com/v1'
LLM_API_KEY = os.environ.get('LLM_API_KEY', '')
LLM_MODEL = 'gpt-4o-mini'
GEN_TEMPERATURE = 0.0
GEN_MAX_TOKENS = 1024
GEN_TIMEOUT = 60

os.environ['CUDA_VISIBLE_DEVICES'] = ''  # Force CPU

print('Environment:', 'Kaggle' if IS_KAGGLE else 'Local')
print(f'FAISS_DIR: {FAISS_DIR}')
print(f'BM25_BASE_DIR: {BM25_BASE_DIR}')
print(f'QA_DIR: {QA_DIR}')

## 3. Verify Paths

In [ ]:
def verify_path(path, desc):
    ok = path.exists()
    print(f'  {"✅" if ok else "❌"} {desc}: {path}')
    return ok

print('=== Kiểm tra paths ===')
ok1 = verify_path(FAISS_DIR, 'FAISS')
ok2 = verify_path(BM25_BASE_DIR, 'BM25')
ok3 = verify_path(QA_DIR, 'QA')

BENCHMARK_PATH = None
if ok3:
    for p in ['qa_final.jsonl', '*.jsonl']:
        found = list(QA_DIR.glob(p))
        if found:
            BENCHMARK_PATH = found[0]
            break
    print(f'  📝 Benchmark: {BENCHMARK_PATH}')

if ok2:
    shards = sorted([d for d in BM25_BASE_DIR.iterdir() if d.is_dir() and d.name.startswith('shard_')])
    print(f'  📦 BM25 shards: {len(shards)}')

if not all([ok1, ok2, ok3, BENCHMARK_PATH]):
    print('\n⚠️ PATHS KHÔNG ĐÚNG!')
    if IS_KAGGLE:
        print('Chạy: !find /kaggle/input -maxdepth 4 -type d')
else:
    print('\n✅ OK!')

## 4. Schema & Metrics

In [ ]:
@dataclass(frozen=True)
class SearchHit:
    point_id: str
    score: float
    payload: dict[str, Any]

@dataclass(frozen=True)
class LatencyBreakdown:
    dense_latency_s: float = 0.0
    sparse_latency_s: float = 0.0
    fusion_latency_s: float = 0.0
    total_latency_s: float = 0.0
    def to_dict(self):
        return {k: round(v, 4) for k, v in {
            'dense_latency_s': self.dense_latency_s,
            'sparse_latency_s': self.sparse_latency_s,
            'fusion_latency_s': self.fusion_latency_s,
            'total_latency_s': self.total_latency_s,
        }.items()}

print('Schema defined.')

In [ ]:
def recall_at_k(ret, rel, k):
    return len(set(ret[:k]) & rel) / len(rel) if rel else 0.0
def hit_at_k(ret, rel, k):
    return 1.0 if rel and set(ret[:k]) & rel else 0.0
def mrr_at_k(ret, rel, k):
    if not rel: return 0.0
    for i, c in enumerate(ret[:k], 1):
        if c in rel: return 1.0/i
    return 0.0
def ndcg_at_k(ret, rel, k):
    if not rel: return 0.0
    dcg = sum(1.0/math.log2(i+1) for i, c in enumerate(ret[:k], 1) if c in rel)
    ideal = sum(1.0/math.log2(i+1) for i in range(1, min(len(rel), k)+1))
    return dcg/ideal if ideal else 0.0
def precision_at_k(ret, rel, k):
    return len(set(ret[:k]) & rel) / k if k else 0.0

def aggregate_metrics(rows, keys):
    out = {'count': len(rows)}
    for k in keys:
        vals = [float(r.get(k, 0)) for r in rows]
        out[k] = sum(vals)/len(vals) if vals else 0.0
    return out

def aggregate_by(rows, field, keys):
    groups = defaultdict(list)
    for r in rows: groups[str(r.get(field) or 'unknown')].append(r)
    return {n: aggregate_metrics(g, keys) for n, g in sorted(groups.items())}

METRIC_NAMES = ['recall', 'hit', 'mrr', 'ndcg', 'precision']
METRIC_KEYS = [f'{n}@{k}' for k in TOP_K_EVAL for n in METRIC_NAMES]

print('Metrics defined.')

## 5. Load QA Benchmark

In [ ]:
print(f'Loading benchmark from {BENCHMARK_PATH} ...')
qa_data = []
with open(BENCHMARK_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line: qa_data.append(json.loads(line))
print(f'Loaded {len(qa_data):,} QA items')

eval_qa = []
skip_un, skip_no = 0, 0
for qa in qa_data:
    at = str(qa.get('answer_type') or '').lower()
    cat = str(qa.get('category') or '').lower()
    if at == 'unanswerable' or cat == 'unanswerable':
        skip_un += 1; continue
    gt = qa.get('ground_truth') or {}
    gt_chunks = {str(c) for c in gt.get('chunk_ids') or [] if c}
    if not gt_chunks:
        skip_no += 1; continue
    eval_qa.append((qa, gt_chunks))

if ABLATION_LIMIT:
    eval_qa = eval_qa[:ABLATION_LIMIT]

print(f'Evaluable: {len(eval_qa)} | Skipped unanswerable: {skip_un} | No GT: {skip_no}')

## 6. Load Dense Retriever (FAISS chunk+metadata)

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

# --- FAISS index ---
print(f'Loading FAISS index...')
t0 = time.perf_counter()
faiss_index = faiss.read_index(str(FAISS_DIR / 'index.faiss'))
print(f'  {faiss_index.ntotal:,} vectors, dim={faiss_index.d} ({time.perf_counter()-t0:.1f}s)')

# --- Payloads ---
print('Loading payloads...')
t0 = time.perf_counter()
dense_payloads = {}
with open(FAISS_DIR / 'payloads.jsonl', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        line = line.strip()
        if line: dense_payloads[i] = json.loads(line)
print(f'  {len(dense_payloads):,} payloads ({time.perf_counter()-t0:.1f}s)')

# --- ID map ---
id_map_path = FAISS_DIR / 'id_map.json'
if id_map_path.exists():
    with open(id_map_path, 'r', encoding='utf-8') as f:
        raw = json.load(f)
    dense_id_map = {int(v): str(k) for k, v in raw.items()}
    print(f'  ID map: {len(dense_id_map):,} entries')
else:
    dense_id_map = {i: str(dense_payloads.get(i, {}).get('chunk_id', i)) for i in dense_payloads}

# --- Embedder ---
print(f'Loading {EMBEDDING_MODEL} ...')
t0 = time.perf_counter()
embedder = SentenceTransformer(EMBEDDING_MODEL, device='cpu')
print(f'  Loaded in {time.perf_counter()-t0:.1f}s')

print('\n✅ Dense retriever ready (chunk+metadata index).')

In [ ]:
# Dense search function
def dense_search(query, *, top_k=30, score_threshold=0.0):
    prefixed = 'query: ' + query
    vec = embedder.encode([prefixed], normalize_embeddings=True)
    qv = np.array(vec, dtype=np.float32)
    limit = min(top_k * 3, faiss_index.ntotal)
    scores, indices = faiss_index.search(qv, limit)
    hits = []
    for sc, idx in zip(scores[0], indices[0]):
        if idx < 0 or float(sc) < score_threshold: continue
        payload = dense_payloads.get(int(idx), {})
        pid = dense_id_map.get(int(idx), str(idx))
        hits.append(SearchHit(point_id=pid, score=float(sc), payload=payload))
        if len(hits) >= top_k: break
    return hits

def dense_search_with_latency(query, **kw):
    t0 = time.perf_counter()
    hits = dense_search(query, **kw)
    return hits, time.perf_counter() - t0

# Smoke test
test_q = 'Điều kiện để người lao động đơn phương chấm dứt hợp đồng lao động'
test_hits, test_lat = dense_search_with_latency(test_q, top_k=5, score_threshold=SCORE_THRESHOLD)
print(f'Dense smoke test: {len(test_hits)} hits in {test_lat:.3f}s')
for r, h in enumerate(test_hits, 1):
    print(f'  [{r}] score={h.score:.4f} id={h.payload.get("chunk_id", h.point_id)}')

## 7. BM25 Sharded Search — Pre-compute ALL queries

Load 1 shard → search tất cả queries → free shard → load shard tiếp.  
Tiết kiệm RAM: chỉ giữ 1 shard trong memory tại mỗi thời điểm.

In [ ]:
# BM25 tokenizer
def simple_tokenize(text):
    text = unicodedata.normalize('NFC', text).lower()
    text = re.sub(r'[^\w\s]', ' ', text, flags=re.UNICODE)
    return [t for t in text.split() if len(t) > 1]

try:
    from underthesea import word_tokenize as _ws
    def bm25_tokenize(text):
        return simple_tokenize(_ws(text, format='text'))
    print('✅ Using underthesea tokenizer')
except ImportError:
    bm25_tokenize = simple_tokenize
    print('⚠️ Using simple tokenizer')

In [ ]:
# ============================================================
# PRE-COMPUTE BM25 RESULTS — SHARD BY SHARD
# ============================================================
# Mỗi shard index độc lập → search từng shard, merge kết quả.
# Load 1 shard tại 1 thời điểm để tiết kiệm RAM.
# ============================================================

shard_dirs = sorted([
    d for d in BM25_BASE_DIR.iterdir()
    if d.is_dir() and d.name.startswith('shard_')
])
print(f'Found {len(shard_dirs)} BM25 shards')

# Pre-tokenize all queries (do once, reuse for all shards)
print('Pre-tokenizing queries...')
all_questions = [str(qa.get('question') or '') for qa, _ in eval_qa]
all_tokenized_queries = [bm25_tokenize(q) for q in all_questions]
print(f'  {len(all_tokenized_queries)} queries tokenized')

# Storage: per-query list of hits across all shards
bm25_all_hits = [[] for _ in range(len(eval_qa))]

total_bm25_t0 = time.perf_counter()

for si, shard_dir in enumerate(shard_dirs):
    shard_t0 = time.perf_counter()
    
    # Load shard BM25 index (pre-built, no retokenization needed)
    idx_path = shard_dir / 'bm25_index.pkl'
    meta_path = shard_dir / 'bm25_metadata.pkl'
    
    if not idx_path.exists() or not meta_path.exists():
        print(f'  ⚠️ Skipping {shard_dir.name}: missing files')
        continue
    
    with idx_path.open('rb') as f:
        bm25 = pickle.load(f)
    with meta_path.open('rb') as f:
        meta = pickle.load(f)
    
    chunk_ids = meta['chunk_ids']
    # Payloads chỉ cần chunk_id — lấy nhẹ, không giữ full payload
    shard_payloads = meta.get('payloads', [{}] * len(chunk_ids))
    del meta  # Free metadata dict
    
    load_time = time.perf_counter() - shard_t0
    
    # Search all queries against this shard
    search_t0 = time.perf_counter()
    for qi, tok_query in enumerate(all_tokenized_queries):
        scores = bm25.get_scores(tok_query)
        top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:TOP_K]
        for idx in top_idx:
            sc = float(scores[idx])
            if sc <= 0: break  # sorted desc, so all after are <= 0
            cid = chunk_ids[idx]
            # Tạo lightweight payload
            pay = {'chunk_id': cid}
            if idx < len(shard_payloads) and isinstance(shard_payloads[idx], dict):
                pay = shard_payloads[idx]
            bm25_all_hits[qi].append(SearchHit(
                point_id=cid, score=sc, payload=pay
            ))
    
    search_time = time.perf_counter() - search_t0
    print(f'  ✅ {shard_dir.name}: {len(chunk_ids):,} docs | load={load_time:.1f}s | search={search_time:.1f}s')
    
    # Free shard memory
    del bm25, chunk_ids, shard_payloads
    gc.collect()

total_bm25_time = time.perf_counter() - total_bm25_t0
print(f'\n✅ BM25 sharded search complete in {total_bm25_time:.1f}s ({total_bm25_time/60:.1f}min)')

# Sort and deduplicate per query
print('Merging and deduplicating results...')
for qi in range(len(eval_qa)):
    # Sort by score descending
    bm25_all_hits[qi].sort(key=lambda h: h.score, reverse=True)
    # Dedup by chunk_id — keep highest score
    seen = set()
    deduped = []
    for h in bm25_all_hits[qi]:
        cid = str(h.payload.get('chunk_id') or h.point_id)
        if cid not in seen:
            seen.add(cid)
            deduped.append(h)
        if len(deduped) >= TOP_K:
            break
    bm25_all_hits[qi] = deduped

# Stats
avg_hits = np.mean([len(h) for h in bm25_all_hits])
print(f'Avg BM25 hits per query: {avg_hits:.1f}')
print(f'\n--- BM25 smoke test (query 0) ---')
for r, h in enumerate(bm25_all_hits[0][:5], 1):
    print(f'  [{r}] score={h.score:.4f} id={h.payload.get("chunk_id", h.point_id)}')

## 8. Corpus Alignment Check

In [ ]:
# Collect BM25 chunk_ids from search results
bm25_found_ids = set()
for hits in bm25_all_hits:
    for h in hits:
        bm25_found_ids.add(str(h.payload.get('chunk_id') or h.point_id))

dense_ids = set(dense_id_map.values())
overlap = dense_ids & bm25_found_ids

all_gt = set()
for _, gt in eval_qa: all_gt.update(gt)

print(f'Dense chunks:  {len(dense_ids):,}')
print(f'BM25 returned: {len(bm25_found_ids):,} unique chunk_ids')
print(f'Overlap:       {len(overlap):,}')
print(f'\nGround truth: {len(all_gt):,}')
print(f'  In Dense: {len(all_gt & dense_ids):,}')
print(f'  In BM25:  {len(all_gt & bm25_found_ids):,}')

## 9. Run Ablation — 3 Configs

In [ ]:
# ============================================================
# HYBRID: merge Dense + BM25 by normalized score
# ============================================================
def hybrid_merge(dense_hits, bm25_hits, top_n=10):
    dense_max = max((h.score for h in dense_hits), default=1.0) or 1.0
    bm25_max = max((h.score for h in bm25_hits), default=1.0) or 1.0
    combined = {}
    for h in dense_hits:
        cid = str(h.payload.get('chunk_id') or h.point_id)
        norm = h.score / dense_max
        if cid not in combined or norm > combined[cid].score:
            combined[cid] = SearchHit(point_id=h.point_id, score=norm, payload=h.payload)
    for h in bm25_hits:
        cid = str(h.payload.get('chunk_id') or h.point_id)
        norm = h.score / bm25_max
        if cid not in combined or norm > combined[cid].score:
            combined[cid] = SearchHit(point_id=h.point_id, score=norm, payload=h.payload)
    return sorted(combined.values(), key=lambda h: h.score, reverse=True)[:top_n]

print('Hybrid merge function defined.')

In [ ]:
# ============================================================
# RUN ALL 3 CONFIGS
# ============================================================

CONFIGS = {
    'Dense-ChunkMeta': {
        'description': 'Dense-only — FAISS (chunk_text + metadata)',
        'mode': 'dense_only',
    },
    'BM25-Only': {
        'description': 'BM25 sparse-only — BM25Okapi sharded search',
        'mode': 'bm25_only',
    },
    'Retrieval-Hybrid-SparseDense': {
        'description': 'Hybrid Dense(chunk+meta) + BM25 — normalized score merge',
        'mode': 'hybrid',
    },
}

all_results = {}
grand_t0 = time.perf_counter()

for cfg_name, cfg in CONFIGS.items():
    print(f'\n{"="*70}')
    print(f'Running: {cfg_name} ({cfg["mode"]})')
    print(f'{"="*70}')
    
    mode = cfg['mode']
    cases, latencies = [], []
    run_t0 = time.perf_counter()
    
    for qi, (qa, gt_chunks) in enumerate(eval_qa):
        question = all_questions[qi]
        qa_id = str(qa.get('qa_id') or qa.get('id') or f'qa_{qi+1}')
        
        try:
            if mode == 'dense_only':
                t0 = time.perf_counter()
                hits, d_lat = dense_search_with_latency(question, top_k=TOP_K, score_threshold=SCORE_THRESHOLD)
                hits = hits[:TOP_N]
                lat = LatencyBreakdown(dense_latency_s=d_lat, total_latency_s=time.perf_counter()-t0)
            
            elif mode == 'bm25_only':
                t0 = time.perf_counter()
                hits = bm25_all_hits[qi][:TOP_N]  # Already computed!
                s_lat = time.perf_counter() - t0
                lat = LatencyBreakdown(sparse_latency_s=s_lat, total_latency_s=s_lat)
            
            else:  # hybrid
                t0 = time.perf_counter()
                d_hits, d_lat = dense_search_with_latency(question, top_k=TOP_K, score_threshold=SCORE_THRESHOLD)
                b_hits = bm25_all_hits[qi]  # Pre-computed
                f_t0 = time.perf_counter()
                hits = hybrid_merge(d_hits, b_hits, top_n=TOP_N)
                f_lat = time.perf_counter() - f_t0
                lat = LatencyBreakdown(
                    dense_latency_s=d_lat, fusion_latency_s=f_lat,
                    total_latency_s=time.perf_counter()-t0
                )
            
            ret_ids = [str(h.payload.get('chunk_id') or h.point_id) for h in hits]
            row = {
                'qa_id': qa_id, 'question': question,
                'category': qa.get('category'), 'difficulty': qa.get('difficulty'),
                'answer_type': qa.get('answer_type'),
                'ground_truth_chunk_ids': sorted(gt_chunks),
                'retrieved_chunk_ids': ret_ids,
                'num_retrieved': len(ret_ids),
            }
            for k in TOP_K_EVAL:
                row[f'recall@{k}'] = recall_at_k(ret_ids, gt_chunks, k)
                row[f'hit@{k}'] = hit_at_k(ret_ids, gt_chunks, k)
                row[f'mrr@{k}'] = mrr_at_k(ret_ids, gt_chunks, k)
                row[f'ndcg@{k}'] = ndcg_at_k(ret_ids, gt_chunks, k)
                row[f'precision@{k}'] = precision_at_k(ret_ids, gt_chunks, k)
            cases.append(row)
            latencies.append(lat.to_dict())
        
        except Exception as e:
            print(f'  ❌ {qa_id}: {e}')
            cases.append({'qa_id': qa_id, 'error': str(e)})
            latencies.append(LatencyBreakdown().to_dict())
        
        if (qi+1) % 50 == 0:
            el = time.perf_counter() - run_t0
            eta = (el/(qi+1)) * (len(eval_qa)-qi-1)
            print(f'  {qi+1}/{len(eval_qa)} ({el:.0f}s elapsed, ~{eta:.0f}s left)')
    
    dur = time.perf_counter() - run_t0
    valid = [c for c in cases if 'error' not in c]
    summary = {
        'config_name': cfg_name, 'config': cfg,
        'counts': {'total': len(cases), 'evaluated': len(valid), 'errors': len(cases)-len(valid)},
        'overall': aggregate_metrics(valid, METRIC_KEYS),
        'by_category': aggregate_by(valid, 'category', METRIC_KEYS),
        'by_difficulty': aggregate_by(valid, 'difficulty', METRIC_KEYS),
        'by_answer_type': aggregate_by(valid, 'answer_type', METRIC_KEYS),
        'latency': {
            'total_run_time_s': round(dur, 2),
            'avg': {k: round(np.mean([l[k] for l in latencies]), 4) for k in latencies[0]} if latencies else {},
            'median': {k: round(float(np.median([l[k] for l in latencies])), 4) for k in latencies[0]} if latencies else {},
        },
    }
    
    print(f'\n  ✅ {cfg_name}: {len(valid)} cases in {dur:.1f}s')
    for m in ['recall@5', 'recall@10', 'mrr@10', 'ndcg@10', 'hit@10']:
        print(f'     {m}: {summary["overall"].get(m, 0):.4f}')
    
    all_results[cfg_name] = (cases, latencies, summary)

grand_total = time.perf_counter() - grand_t0
print(f'\n{"="*70}')
print(f'✅ All done! {grand_total:.0f}s ({grand_total/60:.1f}min)')
print(f'{"="*70}')

## 10. Save Results

In [ ]:
for cfg_name, (cases, latencies, summary) in all_results.items():
    d = OUTPUT_BASE / cfg_name
    d.mkdir(parents=True, exist_ok=True)
    
    manifest = {
        'run_id': cfg_name, 'config_name': cfg_name,
        'benchmark_path': str(BENCHMARK_PATH),
        'index_path': str(FAISS_DIR), 'sparse_index_path': str(BM25_BASE_DIR),
        'retriever_config': {
            'type': summary['config']['mode'],
            'embedding_model': EMBEDDING_MODEL,
            'embedding_input': 'chunk_text + metadata',
            'top_k': TOP_K, 'top_n': TOP_N, 'score_threshold': SCORE_THRESHOLD,
        },
        'timestamp': datetime.now(timezone.utc).isoformat(),
    }
    with (d/'manifest.json').open('w', encoding='utf-8') as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    with (d/'retrieval_cases.jsonl').open('w', encoding='utf-8') as f:
        for c in cases: f.write(json.dumps(c, ensure_ascii=False, separators=(',',':'))+'\n')
    with (d/'retrieval_metrics.json').open('w', encoding='utf-8') as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    with (d/'latency.json').open('w', encoding='utf-8') as f:
        json.dump({'config': cfg_name, 'avg': summary['latency']['avg'],
                   'median': summary['latency']['median']}, f, indent=2)
    print(f'✅ {d}/')

print(f'\nAll saved to {OUTPUT_BASE}/')

## 11. Comparison Tables

In [ ]:
import pandas as pd
from IPython.display import display

KEY = ['recall@1','recall@5','recall@10','hit@5','hit@10','mrr@10','ndcg@10','precision@10']
rows = []
for cn, (_,_,s) in all_results.items():
    row = {'Config': cn}
    for k in KEY: row[k] = round(s['overall'].get(k,0), 4)
    row['Latency(s)'] = round(s['latency']['avg'].get('total_latency_s',0), 4)
    rows.append(row)
df = pd.DataFrame(rows)
print('=== COMPARISON ===')
display(df)
df.to_csv(OUTPUT_BASE/'comparison.csv', index=False)
print(f'Saved to {OUTPUT_BASE}/comparison.csv')

In [ ]:
# By category
for cn, (_,_,s) in all_results.items():
    print(f'\n--- {cn}: By Category ---')
    cr = [{'Cat': c, 'N': m['count'], **{k: round(m.get(k,0),4) for k in ['recall@10','mrr@10','ndcg@10']}}
          for c, m in s.get('by_category', {}).items()]
    if cr: display(pd.DataFrame(cr))

## 12. Charts

In [ ]:
import matplotlib.pyplot as plt

cnames = list(all_results.keys())
short = ['Dense\n(Chunk+Meta)', 'BM25\nOnly', 'Hybrid\n(Dense+BM25)']
x = np.arange(len(cnames))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Retrieval Ablation — Dense vs BM25 vs Hybrid', fontsize=14, fontweight='bold')

# Recall@k
ax = axes[0]
w = 0.18
for i, k in enumerate(TOP_K_EVAL):
    v = [all_results[c][2]['overall'].get(f'recall@{k}',0) for c in cnames]
    ax.bar(x + i*w, v, w, label=f'R@{k}', alpha=0.85)
ax.set_xticks(x+w*1.5); ax.set_xticklabels(short, fontsize=9)
ax.set_title('Recall@k'); ax.legend(fontsize=8); ax.set_ylim(0,1.05); ax.grid(axis='y', alpha=0.3)

# MRR, nDCG, Hit @10
ax = axes[1]
w = 0.22
for i, (m, c) in enumerate([('mrr@10','steelblue'), ('ndcg@10','coral'), ('hit@10','seagreen')]):
    v = [all_results[c2][2]['overall'].get(m,0) for c2 in cnames]
    ax.bar(x+(i-1)*w, v, w, label=m, color=c)
ax.set_xticks(x); ax.set_xticklabels(short, fontsize=9)
ax.set_title('MRR/nDCG/Hit @10'); ax.legend(fontsize=8); ax.set_ylim(0,1.05); ax.grid(axis='y', alpha=0.3)

# Latency
ax = axes[2]
lats = [all_results[c][2]['latency']['avg'].get('total_latency_s',0) for c in cnames]
ax.bar(x, lats, color=['#4e79a7','#f28e2b','#59a14f'])
ax.set_xticks(x); ax.set_xticklabels(short, fontsize=9)
ax.set_title('Avg Latency (s)'); ax.set_ylabel('Seconds'); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(str(OUTPUT_BASE/'charts.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {OUTPUT_BASE}/charts.png')

## 13. Analysis & Conclusions

In [ ]:
sd = all_results['Dense-ChunkMeta'][2]
sb = all_results['BM25-Only'][2]
sh = all_results['Retrieval-Hybrid-SparseDense'][2]

print('='*60)
print('NHẬN XÉT — Người 3 (Kiệt)')
print('='*60)

print('\n1. Dense (chunk+metadata) vs BM25')
print('-'*50)
for m in ['recall@1','recall@5','recall@10','mrr@10','ndcg@10','hit@10']:
    d, b = sd['overall'].get(m,0), sb['overall'].get(m,0)
    w = 'Dense' if d>b else 'BM25' if b>d else 'Tie'
    print(f'  {m:12s}: Dense={d:.4f}  BM25={b:.4f}  diff={d-b:+.4f} → {w}')

print('\n2. Hybrid vs Dense-only')
print('-'*50)
for m in ['recall@1','recall@5','recall@10','mrr@10','ndcg@10']:
    d, h = sd['overall'].get(m,0), sh['overall'].get(m,0)
    pct = ((h-d)/d*100) if d else 0
    print(f'  {m:12s}: Dense={d:.4f}  Hybrid={h:.4f}  diff={h-d:+.4f} ({pct:+.1f}%)')

print('\n3. Best config?')
print('-'*50)
best = max(all_results, key=lambda c: all_results[c][2]['overall'].get('recall@10',0))
print(f'  Best R@10: {best} ⭐')
print(f'  {"Config":40s} {"R@10":>7s} {"MRR@10":>7s} {"Lat":>7s}')
for c in all_results:
    r = all_results[c][2]['overall'].get('recall@10',0)
    m = all_results[c][2]['overall'].get('mrr@10',0)
    l = all_results[c][2]['latency']['avg'].get('total_latency_s',0)
    print(f'  {c:40s} {r:>7.4f} {m:>7.4f} {l:>6.4f}s')

print('\n' + '='*60)

In [ ]:
# Save combined summary
combined = {
    'owner': 'Người 3 (Kiệt)',
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'description': 'Dense(chunk+meta), BM25, Hybrid — No Reranker',
    'settings': {
        'embedding_model': EMBEDDING_MODEL, 'top_k': TOP_K, 'top_n': TOP_N,
        'benchmark': str(BENCHMARK_PATH), 'eval_count': len(eval_qa),
    },
    'configs': {cn: {'overall': s['overall'], 'latency': s['latency'],
                     'by_category': s['by_category']}
               for cn, (_,_,s) in all_results.items()},
}
with (OUTPUT_BASE/'summary.json').open('w', encoding='utf-8') as f:
    json.dump(combined, f, ensure_ascii=False, indent=2)
print(f'Summary: {OUTPUT_BASE}/summary.json')
print('✅ Done!')

## 14. Download (Kaggle)

In [ ]:
import shutil
if IS_KAGGLE:
    shutil.make_archive('/kaggle/working/ablation_results', 'zip', str(OUTPUT_BASE))
    print('✅ /kaggle/working/ablation_results.zip')

for p in sorted(OUTPUT_BASE.rglob('*')):
    if p.is_file(): print(f'  {p.relative_to(OUTPUT_BASE)} ({p.stat().st_size/1024:.0f}KB)')